<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

## 🔑 `state["messages"] = [{...}]` 这一行到底在干什么

**一句话定位**:子 agent 的「干净大脑」是怎么被换上去的。

</div>

<div style="background:#e8f4fd; padding:12px 24px; border-left:4px solid #0366d6; border-radius:4px; width:97%;">

**问题来源**:在 `task_tool.py` 的 `task()` 函数里,有这么一行:

```python
state["messages"] = [{"role": "user", "content": description}]
```

课程说这是 **context 隔离** 的核心一行,但乍一看很普通。它到底在做什么?为什么这一行就实现了「隔离」?

</div>

<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

### 🔍 第一步:字面拆解

</div>

| 元素 | 含义 |
|---|---|
| `state` | 父 agent 通过 `InjectedState` 注入进来的**完整状态**(包含父的所有消息历史) |
| `state["messages"]` | 父 agent 至今为止的**全部对话**(user / AI / tool 来回十几条) |
| `= [{...}]` | **整个列表被替换**为只有 1 条消息的新列表 |
| `description` | 父 agent 调用 `task` 工具时写的**任务说明**(LLM 生成的一句话) |

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

**一行话**:把父 agent 的整段聊天记录,**整个抹掉**,只留下「这是你的任务」这一条消息。

</div>

<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

### 🎯 第二步:什么被擦了 / 什么被保留了

</div>

| State 字段 | 子 agent 看到的 | 为什么 |
|---|---|---|
| `messages` | ❌ 只有 1 条 `description` | 这行代码**主动覆盖** |
| `files` | ✅ 继承父的全部文件 | 没动它,子可以读父写的文件 |
| `todos` | ✅ 继承父的 TODO 列表 | 同上 |

<div style="background:#fff3b0; padding:12px 24px; border-left:4px solid #f0ad4e; border-radius:4px; width:97%;">

**关键洞察**:「隔离」是**精准的**——只隔离对话,不隔离文件 / 任务清单。

父 agent 的「**脑中记忆**」擦掉,但「**硬盘**」还在。

</div>

<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

### 💡 第三步:类比 — 外包给设计师

</div>

| 不隔离(错误做法) | 隔离(本课做法) |
|---|---|
| 把客户跟你 3 小时来回扯皮的微信记录全甩给设计师 | 只发一份《需求说明书.pdf》给设计师 |
| 设计师被无关信息干扰,token 浪费,容易跑偏 | 设计师专注做事,清爽完成 |
| 设计师做完发回的也是一大坨 | 设计师只回一个「做完了 + 结果」 |

<div style="background:#e8f4fd; padding:12px 24px; border-left:4px solid #0366d6; border-radius:4px; width:97%;">

`description` 就是那份**需求说明书** —— 父 agent 已经替子 agent「**翻译**」好了任务。

</div>

<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

### 🔬 第四步:不这样做会怎样(反例)

</div>

如果**删掉这一行**,直接 `sub_agent.invoke(state)`,子 agent 会看到父的全部历史:

<pre style="background:#f6f8fa; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
# 子 agent 启动时看到的 messages:
[
    HumanMessage("用户原始问题"),
    AIMessage("我先调 tavily..."),
    ToolMessage("tavily 结果一大堆"),
    AIMessage("我再写个 todo..."),
    # ... 20 条消息 ...
    AIMessage("好,我现在派单给子 agent")
]
</code></pre>

<div style="background:#ffe6e6; padding:10px 24px; border-left:4px solid #d9534f; border-radius:4px; width:97%;">

**后果**:

- 子 agent 被父的思考过程**污染**(context rot)
- token 成本翻倍
- 子可能「**接着父的话往下说**」,而不是独立完成任务

</div>

<div style="background:linear-gradient(90deg,#0366d6,#6f42c1); color:white; padding:20px 32px; border-radius:8px; width:97%;">

### ✨ 一句话带走

</div>

<div style="background:#e7f7e9; padding:10px 24px; border-left:4px solid #28a745; border-radius:4px; width:97%;">

**`state["messages"] = [{...}]` = 给子 agent 换一颗干净的「大脑」,但保留它的「硬盘」(`files` / `todos`)**。

这就是 **context 隔离** 的物理实现 —— 一行代码,治「**乱**」。

</div>